In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

session = get_active_session()

df = session.sql("""
    SELECT *
    FROM CHURN_DB.CUSTOMER_DATA.CHURN_CLEAN
""").to_pandas()

df.columns = df.columns.str.upper()

df["TOTALCHARGES"] = pd.to_numeric(
    df["TOTALCHARGES"],
    errors="coerce"
)

df["CHURN"] = pd.to_numeric(
    df["CHURN"],
    errors="coerce"
)

df = df.dropna(subset=["CHURN"])

df["CHURN"] = df["CHURN"].astype(int)

X = df.drop(
    columns=["CHURN", "CUSTOMERID"],
    errors="ignore"
)

y = df["CHURN"]

numeric_cols = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=3000,
        class_weight="balanced"
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Dataset shape:", df.shape)
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test,
    y_pred,
    labels=[0, 1]
))

In [ ]:
churn_probability = model.predict_proba(X_test)[:, 1]

predictions_df = pd.DataFrame({
    "CUSTOMERID": df.loc[X_test.index, "CUSTOMERID"].values,
    "ACTUAL_CHURN": y_test.values,
    "PREDICTED_CHURN": y_pred,
    "CHURN_PROBABILITY": churn_probability
})

print(predictions_df.head())

In [ ]:
session.write_pandas(
    predictions_df,
    table_name="CHURN_PREDICTIONS",
    database="CHURN_DB",
    schema="CUSTOMER_DATA",
    auto_create_table=True,
    overwrite=True
)
session.sql("""
    SELECT *
    FROM CHURN_DB.CUSTOMER_DATA.CHURN_PREDICTIONS
    LIMIT 10
""").show()

In [ ]:
high_risk = session.sql("""
    SELECT
        CUSTOMERID,
        ACTUAL_CHURN,
        PREDICTED_CHURN,
        CHURN_PROBABILITY
    FROM CHURN_DB.CUSTOMER_DATA.CHURN_PREDICTIONS
    WHERE CHURN_PROBABILITY >= 0.70
    ORDER BY CHURN_PROBABILITY DESC
""")

high_risk.show()

In [ ]:
session.sql("""
    SELECT
        COUNT(*) AS HIGH_RISK_CUSTOMERS
    FROM CHURN_DB.CUSTOMER_DATA.CHURN_PREDICTIONS
    WHERE CHURN_PROBABILITY >= 0.70
""").show()

In [ ]:
session.sql("""
    SELECT
        COUNT(*) AS TOTAL_TEST_CUSTOMERS,
        SUM(ACTUAL_CHURN) AS ACTUAL_CHURNED,
        SUM(PREDICTED_CHURN) AS PREDICTED_CHURNED,
        SUM(
            CASE
                WHEN CHURN_PROBABILITY >= 0.70 THEN 1
                ELSE 0
            END
        ) AS HIGH_RISK_CUSTOMERS,
        ROUND(AVG(CHURN_PROBABILITY) * 100, 2)
            AS AVERAGE_CHURN_PROBABILITY
    FROM CHURN_DB.CUSTOMER_DATA.CHURN_PREDICTIONS
""").show()

In [ ]:
session.sql("""
    CREATE OR REPLACE VIEW
    CHURN_DB.CUSTOMER_DATA.CHURN_SUMMARY AS
    SELECT
        COUNT(*) AS TOTAL_TEST_CUSTOMERS,
        SUM(ACTUAL_CHURN) AS ACTUAL_CHURNED,
        SUM(PREDICTED_CHURN) AS PREDICTED_CHURNED,
        SUM(
            CASE
                WHEN CHURN_PROBABILITY >= 0.70 THEN 1
                ELSE 0
            END
        ) AS HIGH_RISK_CUSTOMERS,
        ROUND(
            AVG(CHURN_PROBABILITY) * 100,
            2
        ) AS AVERAGE_CHURN_PROBABILITY
    FROM CHURN_DB.CUSTOMER_DATA.CHURN_PREDICTIONS
""").collect()

print("CHURN_SUMMARY view created successfully.")

In [ ]:
session.sql("""
    SELECT *
    FROM CHURN_DB.CUSTOMER_DATA.CHURN_SUMMARY
""").show()

In [ ]:
all_X = df.drop(
    columns=["CHURN", "CUSTOMERID"],
    errors="ignore"
)

all_predictions = model.predict(all_X)
all_probabilities = model.predict_proba(all_X)[:, 1]

all_predictions_df = pd.DataFrame({
    "CUSTOMERID": df["CUSTOMERID"].values,
    "ACTUAL_CHURN": df["CHURN"].values,
    "PREDICTED_CHURN": all_predictions,
    "CHURN_PROBABILITY": all_probabilities
})

print("Total customers:", len(all_predictions_df))
print(all_predictions_df.head())
session.write_pandas(
    all_predictions_df,
    table_name="ALL_CUSTOMER_PREDICTIONS",
    database="CHURN_DB",
    schema="CUSTOMER_DATA",
    auto_create_table=True,
    overwrite=True
)
session.sql("""
    SELECT COUNT(*) AS TOTAL_CUSTOMERS
    FROM CHURN_DB.CUSTOMER_DATA.ALL_CUSTOMER_PREDICTIONS
""").show()


In [ ]:
session.sql("""
    SELECT
        CUSTOMERID,
        ACTUAL_CHURN,
        PREDICTED_CHURN,
        ROUND(CHURN_PROBABILITY * 100, 2) AS CHURN_RISK_PERCENT
    FROM CHURN_DB.CUSTOMER_DATA.ALL_CUSTOMER_PREDICTIONS
    WHERE CHURN_PROBABILITY >= 0.70
    ORDER BY CHURN_PROBABILITY DESC
    LIMIT 20
""").show()

In [ ]:
session.sql("""
    SELECT
        COUNT(*) AS TOTAL_CUSTOMERS,
        SUM(PREDICTED_CHURN) AS PREDICTED_CHURN_CUSTOMERS,
        SUM(
            CASE
                WHEN CHURN_PROBABILITY >= 0.70 THEN 1
                ELSE 0
            END
        ) AS HIGH_RISK_CUSTOMERS,
        ROUND(AVG(CHURN_PROBABILITY) * 100, 2)
            AS AVG_CHURN_RISK_PERCENT
    FROM CHURN_DB.CUSTOMER_DATA.ALL_CUSTOMER_PREDICTIONS
""").show()

In [ ]:
session.sql("""
    CREATE OR REPLACE VIEW
    CHURN_DB.CUSTOMER_DATA.HIGH_RISK_CUSTOMERS AS
    SELECT
        CUSTOMERID,
        ACTUAL_CHURN,
        PREDICTED_CHURN,
        CHURN_PROBABILITY
    FROM CHURN_DB.CUSTOMER_DATA.ALL_CUSTOMER_PREDICTIONS
    WHERE CHURN_PROBABILITY >= 0.70
""").collect()

print("HIGH_RISK_CUSTOMERS view created successfully.")

In [ ]:
session.sql("""
    SELECT
        D.CONTRACT,
        COUNT(*) AS TOTAL_CUSTOMERS,
        SUM(P.ACTUAL_CHURN) AS ACTUAL_CHURNED,
        SUM(P.PREDICTED_CHURN) AS PREDICTED_CHURNED,
        SUM(
            CASE
                WHEN P.CHURN_PROBABILITY >= 0.70 THEN 1
                ELSE 0
            END
        ) AS HIGH_RISK_CUSTOMERS,
        ROUND(
            AVG(P.CHURN_PROBABILITY) * 100,
            2
        ) AS AVG_CHURN_RISK_PERCENT
    FROM CHURN_DB.CUSTOMER_DATA.ALL_CUSTOMER_PREDICTIONS P
    JOIN CHURN_DB.CUSTOMER_DATA.CHURN_CLEAN D
        ON P.CUSTOMERID = D.CUSTOMERID
    GROUP BY D.CONTRACT
    ORDER BY HIGH_RISK_CUSTOMERS DESC
""").show()

In [ ]:
session.sql("""
    SELECT
        D.PAYMENTMETHOD,
        COUNT(*) AS TOTAL_CUSTOMERS,
        SUM(P.ACTUAL_CHURN) AS ACTUAL_CHURNED,
        SUM(P.PREDICTED_CHURN) AS PREDICTED_CHURNED,
        SUM(
            CASE
                WHEN P.CHURN_PROBABILITY >= 0.70 THEN 1
                ELSE 0
            END
        ) AS HIGH_RISK_CUSTOMERS,
        ROUND(
            AVG(P.CHURN_PROBABILITY) * 100,
            2
        ) AS AVG_CHURN_RISK_PERCENT
    FROM CHURN_DB.CUSTOMER_DATA.ALL_CUSTOMER_PREDICTIONS P
    JOIN CHURN_DB.CUSTOMER_DATA.CHURN_CLEAN D
        ON P.CUSTOMERID = D.CUSTOMERID
    GROUP BY D.PAYMENTMETHOD
    ORDER BY HIGH_RISK_CUSTOMERS DESC
""").show()

In [ ]:
session.sql("""
    SELECT
        COUNT(*) AS TOTAL_TEST_CUSTOMERS,
        SUM(ACTUAL_CHURN) AS ACTUAL_CHURNED,
        SUM(PREDICTED_CHURN) AS PREDICTED_CHURNED,
        SUM(
            CASE
                WHEN CHURN_PROBABILITY >= 0.70 THEN 1
                ELSE 0
            END
        ) AS HIGH_RISK_CUSTOMERS,
        ROUND(
            AVG(CHURN_PROBABILITY) * 100,
            2
        ) AS AVG_CHURN_PROBABILITY_PERCENT
    FROM CHURN_DB.CUSTOMER_DATA.CHURN_PREDICTIONS
""").show()